# Afinador Digital de Guitarra con FFT
**Análisis Numérico 1 — Ciclo 01-2026 | UCA**

El notebook está dividido en estas partes:
1. Carga de los archivos `.wav` grabados
2. Preprocesamiento: detección del onset y ventaneo de Hanning
3. DFT manual vs FFT — comparación de tiempos
4. Detección de la frecuencia fundamental con HPS
5. Resultados y gráficas del espectro por cuerda

## Imports y parámetros

In [ ]:
import numpy as np
import scipy.io.wavfile as wav
import matplotlib.pyplot as plt
import time
import os

# fs = 44100 Hz es el estándar de audio; N = 2^14 da Δf ≈ 2.69 Hz (ec. 7)
FS_ESPERADO = 44100
N_BLOQUE    = 16384
F_MIN       = 60    # Hz — límite inferior para filtrar DC y ruido bajo
F_MAX       = 600   # Hz — por encima de esto no hay fundamentales de guitarra

# Frecuencias de las 6 cuerdas al aire en afinación estándar (Tabla 1 del reporte)
CUERDAS = {
    'E2 (6ª - Mi grave)' : 82.41,
    'A2 (5ª - La)'       : 110.00,
    'D3 (4ª - Re)'       : 146.83,
    'G3 (3ª - Sol)'      : 196.00,
    'B3 (2ª - Si)'       : 246.94,
    'E4 (1ª - Mi agudo)' : 329.63,
}

print(f'Δf = {FS_ESPERADO}/{N_BLOQUE} = {FS_ESPERADO/N_BLOQUE:.4f} Hz')

## Funciones de procesamiento

In [ ]:
def cargar_wav(ruta):
    fs, datos = wav.read(ruta)
    if datos.ndim == 2:
        datos = datos.mean(axis=1)
    datos = datos.astype(np.float64)
    datos /= np.max(np.abs(datos))
    return fs, datos


def ventana_hanning(N):
    # ec. (8) del reporte
    n = np.arange(N)
    return 0.5 * (1 - np.cos(2 * np.pi * n / (N - 1)))


def encontrar_onset(datos, umbral=0.05):
    # Los audios grabados tienen silencio al inicio; buscamos la primera muestra
    # que supera el 5% de la amplitud máxima para no analizar el silencio.
    return np.argmax(np.abs(datos) > umbral)


def preprocesar_senal(datos, N=N_BLOQUE):
    # ec. (9): x_mod[n] = x[n] · w[n]
    inicio = encontrar_onset(datos)
    if inicio + N <= len(datos):
        bloque = datos[inicio:inicio + N]
    else:
        bloque = np.zeros(N)
        bloque[:len(datos) - inicio] = datos[inicio:]
    return bloque * ventana_hanning(N)


def dft_manual(x):
    # ec. (3) — implementación directa O(N²), solo para N pequeño
    N = len(x)
    X = np.zeros(N, dtype=complex)
    for k in range(N):
        for n in range(N):
            X[k] += x[n] * np.exp(-1j * 2 * np.pi * k * n / N)
    return X


def calcular_fft(x):
    return np.fft.fft(x)


def espectro_magnitud(X, N):
    # ec. (10): |X[k]| — solo la mitad positiva porque la señal es real
    return np.abs(X[:N // 2])


def espectro_producto_armonico(magnitudes, n_armonicos=4):
    # El HPS resuelve el problema del "tono implícito" (sección 4 del reporte):
    # en cuerdas graves, el 2° armónico a veces tiene más energía que la fundamental.
    # Multiplicar el espectro por versiones decimadas refuerza la fundamental.
    hps = magnitudes.copy()
    for h in range(2, n_armonicos + 1):
        dec = magnitudes[::h]
        L = min(len(hps), len(dec))
        hps[:L] *= dec[:L]
    return hps


def detectar_frecuencia(magnitudes, fs, N, f_min=F_MIN, f_max=F_MAX):
    delta_f = fs / N
    freqs   = np.arange(N // 2) * delta_f
    hps     = espectro_producto_armonico(magnitudes)
    mascara = (freqs >= f_min) & (freqs <= f_max)
    k_target    = np.argmax(hps[mascara])   # ec. (11)
    f_detectada = freqs[mascara][k_target]
    return f_detectada, freqs, magnitudes


def nota_mas_cercana(f_detectada):
    mejor_nota  = min(CUERDAS, key=lambda n: abs(CUERDAS[n] - f_detectada))
    f_teorica   = CUERDAS[mejor_nota]
    error_hz    = f_detectada - f_teorica
    error_cents = 1200 * np.log2(f_detectada / f_teorica)
    return mejor_nota, f_teorica, error_hz, error_cents


print('Funciones listas.')

## DFT manual vs FFT — comparación de tiempos

Usamos una señal de prueba pequeña (N=512) para no esperar demasiado con la DFT.

In [ ]:
fs_test = 44100
t_test  = np.arange(512) / fs_test
x_test  = np.sin(2 * np.pi * 440 * t_test)

t0    = time.time()
X_dft = dft_manual(x_test)
t_dft = time.time() - t0

t0    = time.time()
X_fft = calcular_fft(x_test)
t_fft = time.time() - t0

N = len(x_test)
print(f'N = {N} muestras')
print(f'DFT manual : {t_dft:.4f} s  ({N**2:,} operaciones)')
print(f'FFT numpy  : {t_fft:.6f} s  ({int(N * np.log2(N)):,} operaciones)')
print(f'Factor de aceleración: {t_dft/t_fft:.0f}x')
print(f'\nDiferencia entre espectros: {np.max(np.abs(np.abs(X_dft) - np.abs(X_fft))):.2e}  (ambas dan el mismo resultado)')

In [ ]:
N_test     = len(x_test)
freqs_test = np.arange(N_test // 2) * fs_test / N_test

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, X, titulo in zip(axes,
                          [X_dft, X_fft],
                          ['DFT manual — O(N²)', 'FFT numpy — O(N log₂N)']):
    ax.plot(freqs_test, np.abs(X[:N_test // 2]), color='steelblue')
    ax.axvline(440, color='red', linestyle='--', label='440 Hz')
    ax.set_xlim(0, 1000)
    ax.set_xlabel('Frecuencia [Hz]')
    ax.set_ylabel('Magnitud')
    ax.set_title(titulo)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('DFT vs FFT — señal sintética a 440 Hz', fontsize=13)
plt.tight_layout()
plt.show()

## Efecto del ventaneo de Hanning

Si la frecuencia de la señal no cae exactamente en un bin de la FFT, la ventana rectangular genera *spectral leakage* — la energía se derrama a bins vecinos. La ventana de Hanning lo reduce.

In [ ]:
N_demo  = 1024
fs_demo = 44100
f_demo  = 261.7  # no cae en un bin exacto → provoca leakage
t_demo  = np.arange(N_demo) / fs_demo
x_demo  = np.sin(2 * np.pi * f_demo * t_demo)

X_rect    = np.fft.fft(x_demo)
X_hanning = np.fft.fft(x_demo * ventana_hanning(N_demo))

freqs_demo = np.arange(N_demo // 2) * fs_demo / N_demo

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, X, titulo, color in zip(
    axes,
    [X_rect, X_hanning],
    ['Ventana rectangular — leakage visible', 'Ventana de Hanning — leakage reducido'],
    ['tomato', 'seagreen']
):
    mag_db = 20 * np.log10(np.abs(X[:N_demo // 2]) + 1e-10)
    ax.plot(freqs_demo, mag_db, color=color, linewidth=0.8)
    ax.axvline(f_demo, color='navy', linestyle='--', label=f'{f_demo} Hz')
    ax.set_xlim(100, 500)
    ax.set_ylim(-60, 70)
    ax.set_xlabel('Frecuencia [Hz]')
    ax.set_ylabel('Magnitud [dB]')
    ax.set_title(titulo)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Spectral leakage: ventana rectangular vs Hanning (ec. 8)', fontsize=13)
plt.tight_layout()
plt.show()

## Análisis de los audios grabados

In [ ]:
ARCHIVOS_WAV = [
    'audioswav/6(gruesa).wav',  # Mi E2
    'audioswav/5.wav',          # La A2
    'audioswav/4.wav',          # Re D3
    'audioswav/3.wav',          # Sol G3
    'audioswav/2.wav',          # Si B3
    'audioswav/1(delgada).wav', # Mi E4
]

for f in ARCHIVOS_WAV:
    estado = 'encontrado' if os.path.exists(f) else 'NO encontrado'
    print(f'{f:35s} → {estado}')

## Detección de frecuencia por cuerda

In [ ]:
def analizar_cuerda(ruta_wav, N=N_BLOQUE):
    fs, datos   = cargar_wav(ruta_wav)
    x_mod       = preprocesar_senal(datos, N)
    X           = calcular_fft(x_mod)
    mags        = espectro_magnitud(X, N)
    f_det, freqs, _ = detectar_frecuencia(mags, fs, N)
    nota, f_teo, err_hz, err_cents = nota_mas_cercana(f_det)
    return {
        'archivo'    : ruta_wav,
        'fs'         : fs,
        'freqs'      : freqs,
        'magnitudes' : mags,
        'f_detectada': f_det,
        'nota'       : nota,
        'f_teorica'  : f_teo,
        'error_hz'   : err_hz,
        'error_cents': err_cents,
    }


resultados = []
for archivo in ARCHIVOS_WAV:
    if os.path.exists(archivo):
        r = analizar_cuerda(archivo)
        resultados.append(r)
        estado = 'AFINADA' if abs(r['error_cents']) < 10 else 'DESAFINADA'
        print(f'{archivo}')
        print(f'  Detectada : {r["f_detectada"]:.2f} Hz')
        print(f'  Teorica   : {r["f_teorica"]} Hz  ({r["nota"]})')
        print(f'  Error     : {r["error_hz"]:+.2f} Hz  ({r["error_cents"]:+.1f} cents)  →  {estado}')
        print()

## Espectro de magnitudes por cuerda

In [ ]:
if not resultados:
    print('No hay archivos WAV. Agrega los audios y corre la celda anterior.')
else:
    fig, axes = plt.subplots(len(resultados), 1, figsize=(14, 4 * len(resultados)))
    if len(resultados) == 1:
        axes = [axes]

    for ax, r in zip(axes, resultados):
        mascara = (r['freqs'] >= F_MIN) & (r['freqs'] <= F_MAX)

        ax.plot(r['freqs'], r['magnitudes'], color='lightgray', linewidth=0.8, label='espectro completo')
        ax.plot(r['freqs'][mascara], r['magnitudes'][mascara], color='steelblue', linewidth=1.2, label='60–600 Hz')
        ax.axvline(r['f_detectada'], color='red',   linewidth=1.5, label=f'detectada: {r["f_detectada"]:.2f} Hz')
        ax.axvline(r['f_teorica'],   color='green', linewidth=1.5, linestyle='--',
                   label=f'teorica: {r["f_teorica"]} Hz ({r["nota"]})')

        ax.set_xlim(0, 700)
        ax.set_xlabel('Frecuencia [Hz]')
        ax.set_ylabel('Magnitud')
        ax.set_title(f'{r["archivo"]}  |  error: {r["error_hz"]:+.2f} Hz ({r["error_cents"]:+.1f} cents)')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)

    plt.suptitle('Espectro FFT — cuerdas de guitarra grabadas', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## Tabla resumen

In [ ]:
if resultados:
    print(f'{"Archivo":<30} {"Nota":<22} {"f teorica":>10} {"f detectada":>12} {"Error Hz":>10} {"Cents":>8}  Estado')
    print('-' * 100)
    for r in resultados:
        estado = 'AFINADA' if abs(r['error_cents']) < 10 else 'DESAFINADA'
        print(f'{r["archivo"]:<30} {r["nota"]:<22} {r["f_teorica"]:>10.2f} {r["f_detectada"]:>12.2f} {r["error_hz"]:>+10.2f} {r["error_cents"]:>+8.1f}  {estado}')
else:
    print('Sin resultados.')

## Señales sintéticas — demostración sin archivos de audio

Generamos señales artificiales con armónicos para probar el algoritmo independientemente de los WAV.

In [ ]:
def generar_cuerda_sintetica(f0, fs=44100, N=N_BLOQUE, ruido=0.02):
    # Suma de armónicos con amplitudes decrecientes, más ruido gaussiano leve
    t      = np.arange(N) / fs
    signal = np.zeros(N)
    for i, amp in enumerate([1.0, 0.5, 0.25, 0.15, 0.08], start=1):
        if i * f0 < fs / 2:
            signal += amp * np.sin(2 * np.pi * i * f0 * t)
    signal += ruido * np.random.randn(N)
    return signal


print(f'{"Cuerda":<22} {"f teorica":>10} {"f detectada":>12} {"Error Hz":>10} {"Cents":>8}')
print('-' * 65)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, (nombre, f0) in zip(axes, CUERDAS.items()):
    x_sint = generar_cuerda_sintetica(f0)
    x_mod  = x_sint * ventana_hanning(N_BLOQUE)
    X      = calcular_fft(x_mod)
    mags   = espectro_magnitud(X, N_BLOQUE)
    f_det, freqs, _ = detectar_frecuencia(mags, FS_ESPERADO, N_BLOQUE)
    _, f_teo, err_hz, err_cents = nota_mas_cercana(f_det)

    print(f'{nombre:<22} {f0:>10.2f} {f_det:>12.2f} {err_hz:>+10.2f} {err_cents:>+8.2f}')

    mascara = (freqs >= F_MIN) & (freqs <= F_MAX)
    ax.plot(freqs[mascara], mags[mascara], color='steelblue', linewidth=1)
    ax.axvline(f_det, color='red',   linewidth=1.5, label=f'{f_det:.1f} Hz')
    ax.axvline(f0,    color='green', linewidth=1.5, linestyle='--', label=f'{f0} Hz')
    ax.set_title(nombre, fontsize=10)
    ax.set_xlabel('Frecuencia [Hz]')
    ax.set_ylabel('Magnitud')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('FFT sobre señales sintéticas (6 cuerdas + armónicos)', fontsize=13)
plt.tight_layout()
plt.show()

## Trade-off: resolución vs latencia vs costo computacional

A mayor N, mejor resolución en frecuencia (Δf más chico), pero más tiempo de captura y más operaciones.

In [ ]:
fs        = 44100
valores_N = [512, 1024, 2048, 4096, 8192, 16384, 32768]

delta_f = [fs / N for N in valores_N]
delta_T = [N / fs for N in valores_N]
ops_dft = [N**2 for N in valores_N]
ops_fft = [N * np.log2(N) for N in valores_N]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(valores_N, delta_f, 'o-', color='steelblue')
axes[0].axhline(2.69, color='red', linestyle='--', label='N=16384 → Δf≈2.69 Hz')
axes[0].set_xlabel('N')
axes[0].set_ylabel('Δf [Hz]')
axes[0].set_title('Resolución espectral  Δf = fs / N  (ec. 7)')
axes[0].set_xscale('log', base=2)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(valores_N, [t * 1000 for t in delta_T], 's-', color='orange')
axes[1].set_xlabel('N')
axes[1].set_ylabel('ΔT [ms]')
axes[1].set_title('Latencia de captura  ΔT = N / fs')
axes[1].set_xscale('log', base=2)
axes[1].grid(True, alpha=0.3)

axes[2].plot(valores_N, ops_dft, '^-', color='tomato',   label='DFT  O(N²)')
axes[2].plot(valores_N, ops_fft, 'D-', color='seagreen', label='FFT  O(N log₂N)')
axes[2].set_xlabel('N')
axes[2].set_ylabel('Operaciones')
axes[2].set_title('Complejidad computacional')
axes[2].set_xscale('log', base=2)
axes[2].set_yscale('log')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Efecto del parámetro N sobre el afinador', fontsize=13)
plt.tight_layout()
plt.show()